# Smart Home Appliance Energy Consumption Prediction


## Problem Statement

Households generate energy-consumption data from appliances, lighting systems, indoor sensors, and outdoor weather conditions. These measurements can be used to understand and predict appliance energy consumption.

This project develops a machine learning regression model to predict household appliance energy consumption based on available sensor, lighting, and weather measurements.


## Objective

The primary objective is to develop and deploy a regression model that predicts appliance energy consumption for a 10-minute observation.

The project workflow includes:

1. Exploratory Data Analysis
2. Train/Test Split
3. Feature Identification
4. Data Preprocessing
5. Gradient Boosting Regression
6. Cross-Validation
7. Hyperparameter Tuning
8. Feature Importance Analysis
9. Feature Selection
10. Final Model Evaluation
11. Model Artifact Creation for Streamlit


## Dataset Overview

### Dataset
**UCI Appliances Energy Prediction**

### Dataset Characteristics
- **Observations:** 19,735
- **Input Features:** 28
- **Target:** `Appliances`
- **Target Unit:** Wh per 10-minute interval
- **Sampling Interval:** 10 minutes
- **Collection Period:** Approximately 4.5 months
- **Missing Values:** None

The dataset contains indoor temperature and humidity measurements, outdoor weather measurements, lighting energy use, and two random variables (`rv1` and `rv2`).

### Dataset Source

UCI Machine Learning Repository:

https://archive.ics.uci.edu/dataset/374/appliances%2Benergy%2Bprediction

Download the `energydata_complete.csv` file and place it in the same directory as this notebook.


## 1. Import Required Libraries


In [ ]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, KFold, cross_validate, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TARGET = "Appliances"
DATA_PATH = "energydata_complete.csv"
MODEL_PATH = "smart_home_appliance_energy_model.pkl"

pd.set_option("display.max_columns", 100)


## 2. Load the Dataset


In [ ]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())


## 3. Dataset Inspection


In [ ]:
print("Dataset Information:")
df.info()

print("\nMissing Values:")
display(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nDescriptive Statistics:")
display(df.describe().T)


## 4. Exploratory Data Analysis


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df[TARGET], bins=40, kde=True)
plt.title("Distribution of Appliance Energy Consumption")
plt.xlabel("Appliances (Wh)")
plt.ylabel("Frequency")
plt.show()


In [ ]:
plt.figure(figsize=(12, 8))
correlation = df.select_dtypes(include=np.number).corr()

sns.heatmap(correlation, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()


### EDA Observations

The exploratory analysis is used to understand the distribution of appliance energy consumption and the relationships between numerical variables.

The correlation analysis provides an initial understanding of linear relationships between the input variables and the target.


## 5. Prepare Features and Target

The `Appliances` column is the target variable.

The `date` column is excluded because it is a datetime field rather than a directly usable numerical measurement.

The `rv1` and `rv2` variables are excluded because they are random variables included in the dataset for regression-model testing and filtering of non-predictive attributes.


In [ ]:
excluded_features = ["date", "rv1", "rv2"]

feature_columns = [
    column for column in df.columns
    if column not in excluded_features + [TARGET]
]

X = df[feature_columns].copy()
y = df[TARGET].copy()

print("Number of input features:", X.shape[1])
print("\nFeatures:")
print(feature_columns)


## 6. Train/Test Split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


## 7. Identify Numerical and Categorical Features


In [ ]:
numerical_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical features:", len(numerical_features))
print(numerical_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)


## 8. Data Preprocessing

All model input features are numerical.

`StandardScaler` is used as the preprocessing step and is kept inside the machine learning pipeline.

Feature selection is also included inside the pipeline so that it is fitted only on the training data during cross-validation.


In [ ]:
preprocessor = StandardScaler()

print("Preprocessing: StandardScaler")
print("Categorical encoding required:", len(categorical_features) > 0)


## 9. Gradient Boosting Regression

Gradient Boosting Regressor is used as the main regression algorithm.

Gradient Boosting builds multiple decision trees sequentially, with each new tree learning from the errors of the previous trees. This allows the model to capture nonlinear relationships between household sensor measurements and appliance energy consumption.


In [ ]:
baseline_pipeline = Pipeline([
    ("preprocessing", StandardScaler()),
    ("feature_selection", SelectKBest(score_func=f_regression, k="all")),
    ("model", GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_STATE
    ))
])

baseline_pipeline.fit(X_train, y_train)

baseline_predictions = baseline_pipeline.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_mse = mean_squared_error(y_test, baseline_predictions)
baseline_rmse = np.sqrt(baseline_mse)
baseline_r2 = r2_score(y_test, baseline_predictions)

print("Baseline Model Performance")
print("MAE :", round(baseline_mae, 2))
print("MSE :", round(baseline_mse, 2))
print("RMSE:", round(baseline_rmse, 2))
print("R²  :", round(baseline_r2, 4))


## 10. Cross-Validation

Five-fold K-Fold cross-validation is used on the training data to evaluate the baseline model more reliably during model development.

The test set remains separate and is reserved for final evaluation.


In [ ]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_results = cross_validate(
    baseline_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2"
    },
    n_jobs=-1
)

print("Mean CV MAE :", round((-cv_results["test_MAE"]).mean(), 2))
print("Mean CV RMSE:", round((-cv_results["test_RMSE"]).mean(), 2))
print("Mean CV R²  :", round(cv_results["test_R2"].mean(), 4))


## 11. Hyperparameter Tuning

`RandomizedSearchCV` is used to tune the Gradient Boosting Regressor.

The search focuses on important Gradient Boosting parameters such as the number of trees, learning rate, tree depth, minimum samples, and subsampling.

The number of selected features is also included in the search.


In [ ]:
param_distributions = {
    "model__n_estimators": [100, 200, 300, 400, 500],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.08, 0.1],
    "model__max_depth": [2, 3, 4, 5],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4, 8],
    "model__subsample": [0.7, 0.85, 1.0],
    "feature_selection__k": [10, 15, 20, 25, "all"]
}

random_search = RandomizedSearchCV(
    estimator=baseline_pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
    error_score="raise"
)

random_search.fit(X_train, y_train)

print("Best Parameters:")
print(random_search.best_params_)

print("\nBest CV RMSE:", round(-random_search.best_score_, 2))


## 12. Final Model


In [ ]:
final_model = random_search.best_estimator_

print("Final model:")
print(final_model)


## 13. Feature Importance


In [ ]:
selector = final_model.named_steps["feature_selection"]
model = final_model.named_steps["model"]

selected_mask = selector.get_support()
selected_features = X_train.columns[selected_mask].tolist()

feature_importance = pd.DataFrame({
    "Feature": selected_features,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

display(feature_importance)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=feature_importance.head(15),
    x="Importance",
    y="Feature"
)
plt.title("Top Feature Importances")
plt.show()


## 14. Feature Selection


In [ ]:
print("Number of selected features:", len(selected_features))
print("\nSelected features:")
print(selected_features)


## 15. Final Test Evaluation


In [ ]:
final_predictions = final_model.predict(X_test)

final_mae = mean_absolute_error(y_test, final_predictions)
final_mse = mean_squared_error(y_test, final_predictions)
final_rmse = np.sqrt(final_mse)
final_r2 = r2_score(y_test, final_predictions)

final_metrics = {
    "MAE": final_mae,
    "MSE": final_mse,
    "RMSE": final_rmse,
    "R2": final_r2
}

print("Final Model Performance")
print("-----------------------")
print("MAE :", round(final_mae, 2), "Wh")
print("MSE :", round(final_mse, 2))
print("RMSE:", round(final_rmse, 2), "Wh")
print("R²  :", round(final_r2, 4))


## 16. Actual vs Predicted Values


In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(y_test, final_predictions, alpha=0.5)

minimum = min(y_test.min(), final_predictions.min())
maximum = max(y_test.max(), final_predictions.max())

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--"
)

plt.xlabel("Actual Appliance Energy (Wh)")
plt.ylabel("Predicted Appliance Energy (Wh)")
plt.title("Actual vs Predicted Appliance Energy")
plt.show()


## 17. Example Prediction


In [ ]:
sample_input = X_test.iloc[[0]].copy()

sample_prediction = final_model.predict(sample_input)[0]
actual_value = y_test.iloc[0]

print("Predicted appliance energy:", round(sample_prediction, 2), "Wh")
print("Actual appliance energy   :", round(actual_value, 2), "Wh")

display(sample_input)


## 18. Save Model Artifact for Deployment


In [ ]:
model_artifact = {
    "model": final_model,
    "features": feature_columns,
    "selected_features": selected_features,
    "target": TARGET,
    "target_unit": "Wh per 10-minute interval",
    "model_name": "Gradient Boosting Regressor",
    "metrics": final_metrics,
    "best_params": random_search.best_params_
}

joblib.dump(model_artifact, MODEL_PATH)

print("Model artifact saved successfully.")
print("File:", MODEL_PATH)
print("Size:", round(os.path.getsize(MODEL_PATH) / 1024, 2), "KB")


## 19. Reload Test


In [ ]:
loaded_artifact = joblib.load(MODEL_PATH)

loaded_model = loaded_artifact["model"]
loaded_features = loaded_artifact["features"]

reload_prediction = loaded_model.predict(
    X_test[loaded_features].iloc[[0]]
)[0]

print("Reload successful.")
print("Reloaded prediction:", round(reload_prediction, 2), "Wh")


## 20. Conclusion

This project developed a machine learning regression system for predicting household appliance energy consumption.

### Final Workflow

**EDA → Train/Test Split → Feature Identification → Preprocessing → Gradient Boosting → Cross-Validation → Hyperparameter Tuning → Feature Importance → Feature Selection → Final Evaluation → Model Artifact**

### Deployment

The trained model and preprocessing workflow are saved in `smart_home_appliance_energy_model.pkl` and can be used as the model artifact for the Streamlit application.

### Dataset Limitation

The dataset represents measurements from a single low-energy house over approximately 4.5 months. Therefore, model performance should be interpreted in the context of this dataset and its collection environment.


## References

1. UCI Machine Learning Repository — Appliances Energy Prediction  
   https://archive.ics.uci.edu/dataset/374/appliances%2Benergy%2Bprediction

2. Candanedo, L. M., Feldheim, V., & Deramaix, D. (2017). *Data driven prediction models of energy use of appliances in a low-energy house*.
